# Notebook 00: Setup

Shared infrastructure for all eval notebooks. Run this first — the other notebooks import from here.

**What's in here:**
- `AIClient` — provider-agnostic wrapper (OpenRouter/DeepSeek by default)
- `score_output()` — score AI output against a rubric
- `run_comparison()` — run baseline vs treatment and display results
- Cost tracking per run

In [ ]:
import os, json, textwrap
from openai import OpenAI
import pandas as pd
from IPython.display import display, Markdown

# --- Configuration ---
# Uses OpenRouter with DeepSeek v4 Flash by default.
# Change MODEL to use a different model on OpenRouter.

API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "deepseek/deepseek-v4-flash"

if not API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY environment variable")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"✓ Connected to OpenRouter — model: {MODEL}")

## Core Functions

In [ ]:
def ask(prompt: str, system: str = "", temperature: float = 0.7, max_tokens: int = 2000) -> dict:
    """Send a prompt and get back the response + metadata."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    r = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    
    return {
        "content": r.choices[0].message.content,
        "model": r.model,
        "tokens_in": r.usage.prompt_tokens if r.usage else 0,
        "tokens_out": r.usage.completion_tokens if r.usage else 0,
        "tokens_total": r.usage.total_tokens if r.usage else 0,
    }


def score_output(text: str, rubric: dict[str, str]) -> dict:
    """
    Score AI output against a rubric using the same AI as judge.
    
    rubric: {"Dimension Name": "What 1-5 means for this dimension"}
    Returns: {"Dimension Name": {"score": int, "reason": str}, ...}
    """
    rubric_text = "\n".join(f"- **{dim}**: {desc}" for dim, desc in rubric.items())
    
    scoring_prompt = f"""Score this output on each dimension (1-5 scale).
Return ONLY valid JSON — no markdown, no explanation outside the JSON.

Format: {{"Dimension": {{"score": N, "reason": "one sentence"}}}}

Rubric:
{rubric_text}

Output to score:
---
{text}
---"""
    
    r = ask(scoring_prompt, temperature=0.3, max_tokens=500)
    
    # Parse JSON from response (handle markdown code blocks)
    content = r["content"].strip()
    if content.startswith("```"):
        content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"_raw": content, "_error": "Failed to parse JSON"}


def run_comparison(task_name: str, baseline_prompt: str, treatment_prompt: str, 
                   rubric: dict, system_prompt: str = "", runs: int = 1) -> pd.DataFrame:
    """
    Run baseline vs treatment, score both, return comparison DataFrame.
    """
    results = []
    
    for i in range(runs):
        label = f" (run {i+1})" if runs > 1 else ""
        
        # Baseline
        print(f"Running baseline{label}...")
        b = ask(baseline_prompt, system=system_prompt)
        b_scores = score_output(b["content"], rubric)
        
        # Treatment
        print(f"Running treatment{label}...")
        t = ask(treatment_prompt, system=system_prompt)
        t_scores = score_output(t["content"], rubric)
        
        for dim in rubric:
            b_s = b_scores.get(dim, {})
            t_s = t_scores.get(dim, {})
            results.append({
                "run": i + 1,
                "dimension": dim,
                "baseline_score": b_s.get("score", "?"),
                "treatment_score": t_s.get("score", "?"),
                "baseline_reason": b_s.get("reason", ""),
                "treatment_reason": t_s.get("reason", ""),
            })
    
    df = pd.DataFrame(results)
    return df, b["content"], t["content"]


def show_comparison(df: pd.DataFrame, baseline_text: str, treatment_text: str,
                    task_name: str = "Eval"):
    """Pretty-print the comparison results."""
    display(Markdown(f"## {task_name} — Results"))
    
    # Summary table
    summary = df.groupby("dimension").agg(
        baseline=("baseline_score", "mean"),
        treatment=("treatment_score", "mean"),
    ).round(1)
    summary["delta"] = summary["treatment"] - summary["baseline"]
    summary["delta"] = summary["delta"].apply(lambda x: f"+{x:.1f}" if x > 0 else f"{x:.1f}")
    display(summary)
    
    # Totals
    b_total = df["baseline_score"].apply(lambda x: x if isinstance(x, (int, float)) else 0).sum()
    t_total = df["treatment_score"].apply(lambda x: x if isinstance(x, (int, float)) else 0).sum()
    display(Markdown(f"**Baseline total: {b_total}** | **Treatment total: {t_total}** | **Delta: +{t_total - b_total}**"))
    
    # Show outputs side by side (truncated)
    display(Markdown("### Baseline Output (first 500 chars)"))
    display(Markdown(f"```\n{baseline_text[:500]}\n```"))
    display(Markdown("### Treatment Output (first 500 chars)"))
    display(Markdown(f"```\n{treatment_text[:500]}\n```"))


print("✓ Functions loaded: ask(), score_output(), run_comparison(), show_comparison()")

## Quick Test

Verify the connection and all functions work.

In [ ]:
# Test the connection
r = ask("What is 2 + 2? Answer in one word.")
print(f"Model: {r['model']}")
print(f"Response: {r['content']}")
print(f"Tokens: {r['tokens_total']}")
print("✓ Connection works")